In [1]:
# Importing modules
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.data.transform import clean_string_series

In [2]:
detailed_path = PATHS['raw_data'] / 'UTF8list-3.tsv'
detailed_reservoirs_pd = pd.read_csv(detailed_path, sep='\t')
detailed_reservoirs_pd.head()

,CODE,NAME,RESERVOIR,X,Y,BASIN,RIVERBED,GOOGLE,OPENSTREETMAP,WIKIDATA,PROVINCE,AUTONOMOUS_COMMUNITY,TYPE,CREST_ELEVATION,DAM_HEIGHT,REPORT
0,9250031,SALLENTE,SALLENTE,"42,5003709690001","0,991459934000034",EBRO,BARRANC DE LA LORA,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos de pantalla asfáltica,1770,89,https://sig.mapama.gob.es/WebServices/clientew...
1,3450047,CAZALEGAS,Cazalegas Dam,"40,01298709","-4,70624804799996",TAJO,RÍO ALBERCHE,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,Toledo,Castilla - La Mancha,Presa de materiales sueltos de pantalla de hor...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,9250014,LAGO NEGRO,LAGO NEGRO,"42,5422861970001","1,04058219400002",EBRO,NaN,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos zonificada o de nú...,2340,11,https://sig.mapama.gob.es/WebServices/clientew...
3,9500034,"TORCAS, LAS",Las Trocas reservoir,"41,294686266","-1,08744698599997",EBRO,RÍO HUERVA,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
4,3190006,"BUJEDA, LA","BUJEDA, LA","40,244500658","-2,83518801499997",TAJO,SIN NOMBRE,NaN,NaN,NaN,Guadalajara,Castilla - La Mancha,Presa de materiales sueltos homogénea,"905,5","39,5",https://sig.mapama.gob.es/WebServices/clientew...


In [3]:
detailed_reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   CODE                  367 non-null    int64 
 1   NAME                  367 non-null    object
 2   RESERVOIR             367 non-null    object
 3   X                     367 non-null    object
 4   Y                     367 non-null    object
 5   BASIN                 367 non-null    object
 6   RIVERBED              359 non-null    object
 7   GOOGLE                115 non-null    object
 8   OPENSTREETMAP         36 non-null     object
 9   WIKIDATA              187 non-null    object
 10  PROVINCE              367 non-null    object
 11  AUTONOMOUS_COMMUNITY  367 non-null    object
 12  TYPE                  367 non-null    object
 13  CREST_ELEVATION       184 non-null    object
 14  DAM_HEIGHT            184 non-null    object
 15  REPORT                367 non-null    ob

### Renaming columns

In [4]:
# Will assign them to their lowercase version
detailed_reservoirs_pd.columns = detailed_reservoirs_pd.columns.str.lower()

### Analyzing Code column

In [5]:
print(f"Out of {len(detailed_reservoirs_pd)} rows, there are {detailed_reservoirs_pd['code'].nunique()} code unique ones.")

Out of 367 rows, there are 367 code unique ones.


In [6]:
# We already have the ID, so this column can be deleted
detailed_reservoirs_pd = detailed_reservoirs_pd.drop(columns=['code'])
detailed_reservoirs_pd.head()

,name,reservoir,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,SALLENTE,SALLENTE,"42,5003709690001","0,991459934000034",EBRO,BARRANC DE LA LORA,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos de pantalla asfáltica,1770,89,https://sig.mapama.gob.es/WebServices/clientew...
1,CAZALEGAS,Cazalegas Dam,"40,01298709","-4,70624804799996",TAJO,RÍO ALBERCHE,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,Toledo,Castilla - La Mancha,Presa de materiales sueltos de pantalla de hor...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,LAGO NEGRO,LAGO NEGRO,"42,5422861970001","1,04058219400002",EBRO,NaN,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos zonificada o de nú...,2340,11,https://sig.mapama.gob.es/WebServices/clientew...
3,"TORCAS, LAS",Las Trocas reservoir,"41,294686266","-1,08744698599997",EBRO,RÍO HUERVA,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
4,"BUJEDA, LA","BUJEDA, LA","40,244500658","-2,83518801499997",TAJO,SIN NOMBRE,NaN,NaN,NaN,Guadalajara,Castilla - La Mancha,Presa de materiales sueltos homogénea,"905,5","39,5",https://sig.mapama.gob.es/WebServices/clientew...


### Converting columns types without Missing Values first

In [7]:
detailed_reservoirs_pd.isna().sum()

name                      0
reservoir                 0
x                         0
y                         0
basin                     0
riverbed                  8
google                  252
openstreetmap           331
wikidata                180
province                  0
autonomous_community      0
type                      0
crest_elevation         183
dam_height              183
report                    0
dtype: int64

#### Name and Reservoir columns

In [8]:
# Cleaning the two first name columns
detailed_reservoirs_pd['name'] = clean_string_series(detailed_reservoirs_pd['name'])
detailed_reservoirs_pd['reservoir'] = clean_string_series(detailed_reservoirs_pd['reservoir'])

In [9]:
# Get those whose values between these columns are different
different_values = detailed_reservoirs_pd[(detailed_reservoirs_pd['name'] == detailed_reservoirs_pd['reservoir']) == False]
different_values.head()

,name,reservoir,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
1,cazalegas,cazalegas dam,"40,01298709","-4,70624804799996",TAJO,RÍO ALBERCHE,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,Toledo,Castilla - La Mancha,Presa de materiales sueltos de pantalla de hor...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
3,torcas,trocas reservoir,"41,294686266","-1,08744698599997",EBRO,RÍO HUERVA,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
8,almoguera,faltos,"40,2743056580001","-2,95882007899996",TAJO,RÍO TAJO,https://www.google.com/search?kgmid=/g/12283wdk,NaN,https://www.wikidata.org/wiki/Q5672943,Guadalajara,Castilla - La Mancha,Presa de fábrica de gravedad (hormigón vibrado),NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
9,bachimana alto,bachimana alto reservoir,"42,782028102","-0,223491101999969",EBRO,RÍO CALDARÉS,NaN,NaN,https://www.wikidata.org/wiki/Q95674644,Huesca,Aragón,Presa de fábrica de mampostería,"2207,09999999999","38,1",https://sig.mapama.gob.es/WebServices/clientew...
10,jarosa,jarosa reservoir,"40,6649699230001","-4,11756322999997",TAJO,ARROYO DE LA JAROSA,NaN,NaN,https://www.wikidata.org/wiki/Q1329216,Madrid,Comunidad de Madrid,Presa de fábrica de gravedad (hormigón vibrado),1088,54,https://sig.mapama.gob.es/WebServices/clientew...


In [10]:
# As reservoir has unsignificant words in their names, we will go for name right now
detailed_reservoirs_pd = detailed_reservoirs_pd.drop(columns=['reservoir'])
detailed_reservoirs_pd.head()

,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,sallente,"42,5003709690001","0,991459934000034",EBRO,BARRANC DE LA LORA,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos de pantalla asfáltica,1770,89,https://sig.mapama.gob.es/WebServices/clientew...
1,cazalegas,"40,01298709","-4,70624804799996",TAJO,RÍO ALBERCHE,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,Toledo,Castilla - La Mancha,Presa de materiales sueltos de pantalla de hor...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,lago negro,"42,5422861970001","1,04058219400002",EBRO,NaN,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos zonificada o de nú...,2340,11,https://sig.mapama.gob.es/WebServices/clientew...
3,torcas,"41,294686266","-1,08744698599997",EBRO,RÍO HUERVA,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
4,bujeda,"40,244500658","-2,83518801499997",TAJO,SIN NOMBRE,NaN,NaN,NaN,Guadalajara,Castilla - La Mancha,Presa de materiales sueltos homogénea,"905,5","39,5",https://sig.mapama.gob.es/WebServices/clientew...


#### Coordinate columns

In [11]:
# They are object type right now
detailed_reservoirs_pd['x'] = detailed_reservoirs_pd['x'].str.replace(',', '.').astype(float)
detailed_reservoirs_pd['y'] = detailed_reservoirs_pd['y'].str.replace(',', '.').astype(float)

In [12]:
detailed_reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   name                  367 non-null    object 
 1   x                     367 non-null    float64
 2   y                     367 non-null    float64
 3   basin                 367 non-null    object 
 4   riverbed              359 non-null    object 
 5   google                115 non-null    object 
 6   openstreetmap         36 non-null     object 
 7   wikidata              187 non-null    object 
 8   province              367 non-null    object 
 9   autonomous_community  367 non-null    object 
 10  type                  367 non-null    object 
 11  crest_elevation       184 non-null    object 
 12  dam_height            184 non-null    object 
 13  report                367 non-null    object 
dtypes: float64(2), object(12)
memory usage: 40.3+ KB


### Handling Missing Values

In [13]:
detailed_reservoirs_pd.head()

,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,sallente,42.500371,0.991460,EBRO,BARRANC DE LA LORA,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos de pantalla asfáltica,1770,89,https://sig.mapama.gob.es/WebServices/clientew...
1,cazalegas,40.012987,-4.706248,TAJO,RÍO ALBERCHE,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,Toledo,Castilla - La Mancha,Presa de materiales sueltos de pantalla de hor...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,lago negro,42.542286,1.040582,EBRO,NaN,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos zonificada o de nú...,2340,11,https://sig.mapama.gob.es/WebServices/clientew...
3,torcas,41.294686,-1.087447,EBRO,RÍO HUERVA,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
4,bujeda,40.244501,-2.835188,TAJO,SIN NOMBRE,NaN,NaN,NaN,Guadalajara,Castilla - La Mancha,Presa de materiales sueltos homogénea,"905,5","39,5",https://sig.mapama.gob.es/WebServices/clientew...


In [14]:
detailed_reservoirs_pd.isna().sum()

name                      0
x                         0
y                         0
basin                     0
riverbed                  8
google                  252
openstreetmap           331
wikidata                180
province                  0
autonomous_community      0
type                      0
crest_elevation         183
dam_height              183
report                    0
dtype: int64

#### Riverbed Missing Values

In [15]:
# Firstly, we are processing the non-nan ones, so that we can compare them
nan_mask = (detailed_reservoirs_pd['riverbed'].isna() == False)
# We process and save them to the dataframe
detailed_reservoirs_pd.loc[nan_mask, 'riverbed'] = clean_string_series(detailed_reservoirs_pd.loc[nan_mask, 'riverbed'])
# We filter the non-nan ones
non_nan_riverbed = detailed_reservoirs_pd[detailed_reservoirs_pd['riverbed'].isna() == False]
non_nan_riverbed_column = non_nan_riverbed['riverbed']
print(f"There are {len(non_nan_riverbed_column)} riverbeds, and {non_nan_riverbed_column.nunique()} are different")

There are 359 riverbeds, and 242 are different


In [16]:
# Now we get those that are repeated at least 3 times
non_nan_riverbed[non_nan_riverbed_column.map(non_nan_riverbed_column.value_counts() > 2)].sort_values('riverbed')

,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
364,iznajar,37.276153,-4.386873,GUADALQUIVIR,rio genil,NaN,NaN,https://www.wikidata.org/wiki/Q5369447,Córdoba,Andalucía,Presa de fábrica de gravedad (hormigón vibrado),426,"121,599999999999",https://sig.mapama.gob.es/WebServices/clientew...
192,cordobilla,37.349256,-4.721161,GUADALQUIVIR,rio genil,NaN,NaN,NaN,Sevilla,Andalucía,Presa de fábrica de gravedad (hormigón vibrado),"218,25","45,25",https://sig.mapama.gob.es/WebServices/clientew...
250,canales,37.160469,-3.480476,GUADALQUIVIR,rio genil,NaN,NaN,https://www.wikidata.org/wiki/Q1032728,Granada,Andalucía,Presa de materiales sueltos zonificada o de nú...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
33,malpasillo jauja,37.299151,-4.676820,GUADALQUIVIR,rio genil,NaN,NaN,NaN,Córdoba,Andalucía,Presa de fábrica de contrafuertes,250,36,https://sig.mapama.gob.es/WebServices/clientew...
148,sierra boyera,38.260807,-5.221643,GUADALQUIVIR,rio guadiato,NaN,NaN,https://www.wikidata.org/wiki/Q5369459,Córdoba,Andalucía,Presa de materiales sueltos zonificada o de nú...,"503,5","10,25",https://sig.mapama.gob.es/WebServices/clientew...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,minilla,37.664791,-6.185621,GUADALQUIVIR,rivera huelva,NaN,NaN,NaN,Sevilla,Andalucía,Presa de fábrica de gravedad (hormigón vibrado),NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
198,gergal,37.567273,-6.049054,GUADALQUIVIR,rivera huelva,NaN,NaN,NaN,Sevilla,Andalucía,Presa de fábrica de arco-gravedad,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
333,pedrera,38.032751,-0.875544,SEGURA,sin nombre,NaN,NaN,NaN,Alacant/Alicante,Comunitat Valenciana,Presa de materiales sueltos homogénea,"110,54","65,54",https://sig.mapama.gob.es/WebServices/clientew...
4,bujeda,40.244501,-2.835188,TAJO,sin nombre,NaN,NaN,NaN,Guadalajara,Castilla - La Mancha,Presa de materiales sueltos homogénea,"905,5","39,5",https://sig.mapama.gob.es/WebServices/clientew...


In [17]:
# Sin nombre means without name, those are NaNs:
mask = detailed_reservoirs_pd['riverbed'] == 'sin nombre'
detailed_reservoirs_pd.loc[mask, 'riverbed'] = np.nan

In [18]:
# Now we get those that are repeated at least 3 times
non_nan_riverbed = non_nan_riverbed[non_nan_riverbed['riverbed'] != 'sin nombre']
non_nan_riverbed[non_nan_riverbed_column.map(non_nan_riverbed_column.value_counts() > 2)].sort_values('riverbed')

C:\Users\usuario\AppData\Local\Temp\ipykernel_14232\2249824163.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  non_nan_riverbed[non_nan_riverbed_column.map(non_nan_riverbed_column.value_counts() > 2)].sort_values('riverbed')


,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
364,iznajar,37.276153,-4.386873,GUADALQUIVIR,rio genil,NaN,NaN,https://www.wikidata.org/wiki/Q5369447,Córdoba,Andalucía,Presa de fábrica de gravedad (hormigón vibrado),426,"121,599999999999",https://sig.mapama.gob.es/WebServices/clientew...
192,cordobilla,37.349256,-4.721161,GUADALQUIVIR,rio genil,NaN,NaN,NaN,Sevilla,Andalucía,Presa de fábrica de gravedad (hormigón vibrado),"218,25","45,25",https://sig.mapama.gob.es/WebServices/clientew...
250,canales,37.160469,-3.480476,GUADALQUIVIR,rio genil,NaN,NaN,https://www.wikidata.org/wiki/Q1032728,Granada,Andalucía,Presa de materiales sueltos zonificada o de nú...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
33,malpasillo jauja,37.299151,-4.676820,GUADALQUIVIR,rio genil,NaN,NaN,NaN,Córdoba,Andalucía,Presa de fábrica de contrafuertes,250,36,https://sig.mapama.gob.es/WebServices/clientew...
148,sierra boyera,38.260807,-5.221643,GUADALQUIVIR,rio guadiato,NaN,NaN,https://www.wikidata.org/wiki/Q5369459,Córdoba,Andalucía,Presa de materiales sueltos zonificada o de nú...,"503,5","10,25",https://sig.mapama.gob.es/WebServices/clientew...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,rivera gata,40.133155,-6.647638,TAJO,rivera gata,NaN,NaN,NaN,Cáceres,Extremadura,Presa de materiales sueltos homogénea,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
163,rivera gata,40.136397,-6.638712,TAJO,rivera gata,NaN,NaN,NaN,Cáceres,Extremadura,Presa de materiales sueltos zonificada o de nú...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
302,aracena,37.907890,-6.451845,GUADALQUIVIR,rivera huelva,NaN,NaN,NaN,Huelva,Andalucía,Presa de fábrica de contrafuertes,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
180,minilla,37.664791,-6.185621,GUADALQUIVIR,rivera huelva,NaN,NaN,NaN,Sevilla,Andalucía,Presa de fábrica de gravedad (hormigón vibrado),NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...


In [19]:
# Those with the same riverbed are sometimes very close. We will asign the riverbed of the closest one
def haversine(lat1, lon1, lat2, lon2):
    # Haversine formula to calculate the distance between two points on the Earth
    R = 6371  # Radius of the Earth in kilometers
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)

    a = np.sin(delta_phi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [20]:
# Reset the index to avoid issues with the haversine function
non_nan_riverbed = non_nan_riverbed.reset_index(drop=True)
non_nan_riverbed.head()

,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,sallente,42.500371,0.991460,EBRO,barranc lora,NaN,NaN,NaN,Lleida,Cataluña,Presa de materiales sueltos de pantalla asfáltica,1770,89,https://sig.mapama.gob.es/WebServices/clientew...
1,cazalegas,40.012987,-4.706248,TAJO,rio alberche,https://www.google.com/search?kgmid=/g/11dfr4j268,NaN,https://www.wikidata.org/wiki/Q30278314,Toledo,Castilla - La Mancha,Presa de materiales sueltos de pantalla de hor...,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,torcas,41.294686,-1.087447,EBRO,rio huerva,NaN,NaN,https://www.wikidata.org/wiki/Q23991169,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón vibrado),"624,799999999999","39,45",https://sig.mapama.gob.es/WebServices/clientew...
3,torcon,39.625277,-4.385500,TAJO,arroyo torcon,NaN,NaN,NaN,Toledo,Castilla - La Mancha,Presa de fábrica de gravedad (hormigón vibrado),NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...
4,puerto vallehermoso,38.867927,-3.168491,GUADIANA,rio azuer,NaN,NaN,NaN,Ciudad Real,Castilla - La Mancha,Presa de materiales sueltos homogénea,NaN,NaN,https://sig.mapama.gob.es/WebServices/clientew...


In [21]:
# Haversine function that has two series as arguments
def haversine_series(lat1_serie, lon1_serie, lat2, lon2):
    return haversine(lat1_serie.values, lon1_serie.values, lat2, lon2)

In [22]:
# Could have given a fully-vectorized approach using AI, but this is my version
nan_riverbed = detailed_reservoirs_pd[detailed_reservoirs_pd['riverbed'].isna()]
nan_riverbed = nan_riverbed[['x', 'y']]

for index, row in nan_riverbed.iterrows():
    distances = haversine_series(non_nan_riverbed['y'], non_nan_riverbed['x'], row['y'], row['x'])
    closest_index = distances.argmin()
    closest_riverbed = non_nan_riverbed.iloc[closest_index]['riverbed']
    detailed_reservoirs_pd.at[index, 'riverbed'] = closest_riverbed

### Crest Elevation Missing Values

In [23]:
detailed_reservoirs_pd.isna().sum()

name                      0
x                         0
y                         0
basin                     0
riverbed                  0
google                  252
openstreetmap           331
wikidata                180
province                  0
autonomous_community      0
type                      0
crest_elevation         183
dam_height              183
report                    0
dtype: int64

In [24]:
detailed_reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   name                  367 non-null    object 
 1   x                     367 non-null    float64
 2   y                     367 non-null    float64
 3   basin                 367 non-null    object 
 4   riverbed              367 non-null    object 
 5   google                115 non-null    object 
 6   openstreetmap         36 non-null     object 
 7   wikidata              187 non-null    object 
 8   province              367 non-null    object 
 9   autonomous_community  367 non-null    object 
 10  type                  367 non-null    object 
 11  crest_elevation       184 non-null    object 
 12  dam_height            184 non-null    object 
 13  report                367 non-null    object 
dtypes: float64(2), object(12)
memory usage: 40.3+ KB


In [25]:
# We convert non-nan crest elevation to float
mask = detailed_reservoirs_pd['crest_elevation'].isna() == False
detailed_reservoirs_pd.loc[mask, 'crest_elevation'] = detailed_reservoirs_pd.loc[mask, 'crest_elevation'].str.replace(',', '.').astype(float)

In [26]:
# We assign the crest elevation to the non-nan ones as the value from the closest one
# Could have given a fully-vectorized approach using AI, but this is my version
nan_crest_elevation = detailed_reservoirs_pd[detailed_reservoirs_pd['crest_elevation'].isna()]
nan_crest_elevation = nan_crest_elevation[['x', 'y']]
non_nan_crest_elevation = detailed_reservoirs_pd[detailed_reservoirs_pd['crest_elevation'].isna() == False]
non_nan_crest_elevation = non_nan_crest_elevation[['x', 'y', 'crest_elevation']].reset_index(drop=True)

for index, row in nan_crest_elevation.iterrows():
    distances = haversine_series(non_nan_crest_elevation['y'], non_nan_crest_elevation['x'], row['y'], row['x'])
    closest_index = distances.argmin()
    closest_crest_elevation = non_nan_crest_elevation.iloc[closest_index]['crest_elevation']
    detailed_reservoirs_pd.at[index, 'crest_elevation'] = closest_crest_elevation

In [27]:
# Now that we have the crest elevation filled, we can convert it to float
detailed_reservoirs_pd['crest_elevation'] = detailed_reservoirs_pd['crest_elevation'].astype(float)

### Removing duplicated rows

In [28]:
# First sight:
mask_duplicated = detailed_reservoirs_pd['name'].value_counts() > 1
duplicated = detailed_reservoirs_pd.loc[detailed_reservoirs_pd['name'].map(mask_duplicated)].sort_values('name')
duplicated.head(10)

,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
338,aguilar campoo,42.789079,-4.291817,DUERO,rio pisuerga,NaN,NaN,NaN,Palencia,Castilla y León,Presa de materiales sueltos homogénea,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
337,aguilar campoo,42.786754,-4.295204,DUERO,rio pisuerga,NaN,NaN,NaN,Palencia,Castilla y León,Presa de materiales sueltos homogénea,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
336,aguilar campoo,42.795091,-4.287344,DUERO,rio pisuerga,NaN,NaN,NaN,Palencia,Castilla y León,Presa de fábrica de gravedad (hormigón vibrado),840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
102,alsa torina,43.094733,-3.999959,CANTABRICO OCCIDENTAL,rio torina o torino,NaN,NaN,NaN,Cantabria,Cantabria,Presa de fábrica de arco-gravedad,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
101,alsa torina,43.072080,-4.012893,CANTABRICO OCCIDENTAL,arroyo mojon,NaN,NaN,NaN,Cantabria,Cantabria,Presa de materiales sueltos de pantalla asfáltica,840.0,NaN,https://sig.mapama.gob.es/WebServices/clientew...
186,anarbe,43.212302,-1.875245,CANTABRICO ORIENTAL,anarbe ibaia o enobietaku erreka,NaN,NaN,NaN,Gipuzkoa/Guipúzcoa,País Vasco,Presa de fábrica de arco-gravedad,163.5,"79,5",https://sig.mapama.gob.es/WebServices/clientew...
187,anarbe,43.212673,-1.878279,CANTABRICO ORIENTAL,anarbe ibaia o enobietaku erreka,NaN,NaN,NaN,Gipuzkoa/Guipúzcoa,País Vasco,Presa de fábrica de gravedad (hormigón vibrado),162.5,"7,5",https://sig.mapama.gob.es/WebServices/clientew...
53,arcos,36.751339,-5.795583,GUADALETE Y BARBATE,rio guadalete,NaN,NaN,https://www.wikidata.org/wiki/Q17279790,Cádiz,Andalucía,Presa de fábrica de gravedad (hormigón vibrado),108.5,NaN,https://sig.mapama.gob.es/WebServices/clientew...
54,arcos,36.751390,-5.791509,GUADALETE Y BARBATE,rio guadalete,NaN,NaN,https://www.wikidata.org/wiki/Q17279790,Cádiz,Andalucía,Presa de materiales sueltos zonificada o de nú...,108.5,NaN,https://sig.mapama.gob.es/WebServices/clientew...
255,arenos,40.093352,-0.544595,JUCAR,barranc jau,NaN,NaN,https://www.wikidata.org/wiki/Q3376336,Castelló/Castellón,Comunitat Valenciana,Presa de materiales sueltos de pantalla de mat...,603.0,18,https://sig.mapama.gob.es/WebServices/clientew...


In [29]:
# We well keep the first one that has the highest dam height
detailed_reservoirs_pd = detailed_reservoirs_pd.sort_values('dam_height', ascending=False).drop_duplicates('name')

### Dam height Missing Values

In [30]:
# We convert non-nan dam height to float
mask = detailed_reservoirs_pd['dam_height'].isna() == False
detailed_reservoirs_pd.loc[mask, 'dam_height'] = detailed_reservoirs_pd.loc[mask, 'dam_height'].str.replace(',', '.').astype(float)

In [31]:
detailed_reservoirs_pd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 320 entries, 330 to 366
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   name                  320 non-null    object 
 1   x                     320 non-null    float64
 2   y                     320 non-null    float64
 3   basin                 320 non-null    object 
 4   riverbed              320 non-null    object 
 5   google                100 non-null    object 
 6   openstreetmap         29 non-null     object 
 7   wikidata              162 non-null    object 
 8   province              320 non-null    object 
 9   autonomous_community  320 non-null    object 
 10  type                  320 non-null    object 
 11  crest_elevation       320 non-null    float64
 12  dam_height            159 non-null    object 
 13  report                320 non-null    object 
dtypes: float64(3), object(11)
memory usage: 37.5+ KB


### Processing the rest of string data

In [32]:
detailed_reservoirs_pd.head()

,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
330,santa ana,41.882506,0.580850,EBRO,rio noguera ribagorcana,NaN,NaN,NaN,Huesca,Aragón,Presa de fábrica de arco-gravedad,380.30,99.6,https://sig.mapama.gob.es/WebServices/clientew...
124,tanes,43.220950,-5.427761,CANTABRICO OCCIDENTAL,rio nalon,NaN,NaN,NaN,Asturias,Principado de Asturias,Presa de fábrica de gravedad (hormigón vibrado),495.00,95.0,https://sig.mapama.gob.es/WebServices/clientew...
111,val,41.875792,-1.786973,EBRO,rio casa o val,NaN,NaN,https://www.wikidata.org/wiki/Q95570755,Zaragoza,Aragón,Presa de fábrica de gravedad (hormigón compact...,629.15,94.4,https://sig.mapama.gob.es/WebServices/clientew...
266,peares,42.465137,-7.723744,MIÑO-SIL,rio mino,https://www.google.com/search?kgmid=/g/155svy9k,NaN,https://www.wikidata.org/wiki/Q11986800,Lugo,Galicia,Presa de fábrica de gravedad (hormigón vibrado),196.50,94.0,https://sig.mapama.gob.es/WebServices/clientew...
315,mediano,42.313617,0.210738,EBRO,rio cinca,https://www.google.com/search?kgmid=/g/121bdb0g,https://www.openstreetmap.org/relation/7141904,https://www.wikidata.org/wiki/Q3215482,Huesca,Aragón,Presa de fábrica de gravedad (hormigón vibrado),529.50,92.0,https://sig.mapama.gob.es/WebServices/clientew...


In [33]:
detailed_reservoirs_pd['basin'] = clean_string_series(detailed_reservoirs_pd['basin'])
detailed_reservoirs_pd['province'] = clean_string_series(detailed_reservoirs_pd['province'])
detailed_reservoirs_pd['autonomous_community'] = clean_string_series(detailed_reservoirs_pd['autonomous_community'])
detailed_reservoirs_pd['type'] = clean_string_series(detailed_reservoirs_pd['type'])

In [34]:
detailed_reservoirs_pd.head()

,name,x,y,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
330,santa ana,41.882506,0.580850,ebro,rio noguera ribagorcana,NaN,NaN,NaN,huesca,aragon,presa fabrica arco gravedad,380.30,99.6,https://sig.mapama.gob.es/WebServices/clientew...
124,tanes,43.220950,-5.427761,cantabrico occidental,rio nalon,NaN,NaN,NaN,asturias,principado asturias,presa fabrica gravedad (hormigon vibrado),495.00,95.0,https://sig.mapama.gob.es/WebServices/clientew...
111,val,41.875792,-1.786973,ebro,rio casa o val,NaN,NaN,https://www.wikidata.org/wiki/Q95570755,zaragoza,aragon,presa fabrica gravedad (hormigon compactado),629.15,94.4,https://sig.mapama.gob.es/WebServices/clientew...
266,peares,42.465137,-7.723744,mino sil,rio mino,https://www.google.com/search?kgmid=/g/155svy9k,NaN,https://www.wikidata.org/wiki/Q11986800,lugo,galicia,presa fabrica gravedad (hormigon vibrado),196.50,94.0,https://sig.mapama.gob.es/WebServices/clientew...
315,mediano,42.313617,0.210738,ebro,rio cinca,https://www.google.com/search?kgmid=/g/121bdb0g,https://www.openstreetmap.org/relation/7141904,https://www.wikidata.org/wiki/Q3215482,huesca,aragon,presa fabrica gravedad (hormigon vibrado),529.50,92.0,https://sig.mapama.gob.es/WebServices/clientew...


In [35]:
# Save the cleaned dataframe
detailed_reservoirs_pd.to_csv(PATHS['processed_data'] / 'missing_values_imputation.csv', index=False)

cleaned_detailed_reservoirs_path = PATHS['cleaned_data'] / 'detailed_reservoirs_cleaned.csv'
cleaned_detailed_reservoirs_path.parent.mkdir(parents=True, exist_ok=True)
detailed_reservoirs_pd.to_csv(cleaned_detailed_reservoirs_path, index=False)